# K-Means, K-Medoids (PAM), and Silhouette Analysis
Accompanies **Chapter 20: K-Means and Partitioning Around Medoids (PAM)**.

K-Means minimizes Within-Cluster Sum of Squares (Inertia):
$$J = \sum_{k=1}^K \sum_{\mathbf{x}_i \in S_k} \|\mathbf{x}_i - \boldsymbol{\mu}_k\|_2^2$$
Because centroids $\boldsymbol{\mu}_k$ are sample arithmetic means, a single outlier pulls the centroid away from the true cluster mass.

**Partitioning Around Medoids (PAM / K-Medoids)** restricts cluster centers to actual historical observations $\mathbf{m}_k \in \mathcal{D}$, minimizing total dissimilarity across arbitrary distance metrics:
$$J_{\text{medoids}} = \sum_{i=1}^n \min_{\mathbf{m}_k \in \mathcal{M}} D(\mathbf{x}_i, \mathbf{m}_k)$$

This notebook benchmarks K-Means vs. K-Medoids under outlier corruption and sweeps Silhouette coefficients to identify optimal cluster count $K$.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

data_path = os.path.abspath('../../data/bd_train.csv')
df = pd.read_csv(data_path)

features = ['age', 'balance', 'duration', 'campaign']
X = df[features].dropna().sample(n=2000, random_state=42)
X_scaled = StandardScaler().fit_transform(X)

### Sweeping Silhouette Scores for Optimal $K$
The Silhouette Coefficient evaluates cohesion $a(i)$ vs. separation $b(i)$:
$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

In [ ]:
k_range = range(2, 7)
inertias, sil_scores = [], []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(k_range, inertias, marker='o', color='royalblue', linewidth=2)
ax1.set_xlabel('Number of Clusters ($K$)', fontsize=11)
ax1.set_ylabel('Inertia (WCSS)', fontsize=11)
ax1.set_title('The Elbow Method', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, sil_scores, marker='s', color='forestgreen', linewidth=2)
ax2.set_xlabel('Number of Clusters ($K$)', fontsize=11)
ax2.set_ylabel('Mean Silhouette Score', fontsize=11)
ax2.set_title('Silhouette Analysis', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Demonstrating Outlier Insulation: Centroid vs. Medoid
We place an extreme outlier at $(60, 60)$ alongside a Gaussian cluster centered at $(0, 0)$.

In [ ]:
np.random.seed(42)
clean_cluster = np.random.normal(loc=0.0, scale=1.0, size=(100, 2))
contaminated = np.vstack([clean_cluster, [60.0, 60.0]])

# K-Means centroid
km = KMeans(n_clusters=1, n_init=10, random_state=42).fit(contaminated)
kmeans_center = km.cluster_centers_[0]

# K-Medoid center: observation minimizing sum of distances to all points
diff = contaminated[:, np.newaxis, :] - contaminated[np.newaxis, :, :]
dist_matrix = np.linalg.norm(diff, axis=-1)
medoid_idx = np.argmin(np.sum(dist_matrix, axis=1))
medoid_center = contaminated[medoid_idx]

print(f"True cluster center of mass: [0.00, 0.00]")
print(f"K-Means centroid:           [{kmeans_center[0]:.2f}, {kmeans_center[1]:.2f}]  (Shifted by outlier!)")
print(f"K-Medoids medoid:           [{medoid_center[0]:.2f}, {medoid_center[1]:.2f}]  (Anchored to cluster mass!)")